# Notebook 03 — Interview Agent

**Purpose:** After completing simulations, students can take an AI-conducted interview. This is a multi-turn conversation where the AI asks questions, listens to answers, asks follow-up questions, and scores the student at the end.

**Interview types:**
- **Technical** — role-specific coding/concept questions
- **HR** — behavioral questions (why this role, teamwork, conflict resolution)
- **Behavioral** — situation-based questions (STAR format)

**AI Model:** Groq — `llama-3.1-8b-instant` via `langchain-groq`

**Key Techniques:**
- `ConversationBufferWindowMemory` (LangChain) for conversation context
- `StateGraph` (LangGraph) for the interview workflow
- System prompt persona: professional but friendly interviewer
- Sentiment and confidence scoring from transcript

**Input per turn:** `{ interview_type, role, conversation_history, user_message, simulation_context }`

**Output per turn:** `{ ai_response, is_complete, next_action }`

**Output at conclusion:** Scores, feedback, and interview summary

---
## 1. Setup & Dependencies

In [ ]:
# %pip install -r requirements.txt

In [ ]:
import os
import json
import re
from datetime import datetime
from typing import TypedDict, Optional, List
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
# Conversation history managed via LangGraph state
from langgraph.graph import StateGraph, END

load_dotenv()

print("All imports loaded")
print(f"Groq API key present: {bool(os.getenv('GROQ_API_KEY'))}")

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")

MODEL_NAME = "llama-3.1-8b-instant"

model = ChatGroq(
    model=MODEL_NAME,
    temperature=0.7,
    max_tokens=1024,
    api_key=GROQ_API_KEY,
)

print(f"ChatGroq initialized with model: {MODEL_NAME}")

---
## 2. Define Interview State (LangGraph)

In [ ]:
class InterviewState(TypedDict):
    interview_type: str          # "technical" | "hr" | "behavioral"
    role: str                    # e.g. "MERN Stack Developer"
    conversation_history: list   # list of {role: str, content: str}
    simulation_context: str      # skills demonstrated during simulation
    user_message: str            # current student message (empty = start)
    ai_response: str             # current AI response
    turn_count: int
    max_turns: int
    is_complete: bool
    next_action: str             # "continue" | "conclude"
    scores: Optional[dict]
    overall_score: Optional[int]
    feedback: Optional[str]
    interview_summary: Optional[str]
    error: Optional[str]

print("InterviewState defined")

---
## 3. Load Career Knowledge & Prompts

In [ ]:
DATA_DIR = os.path.join(os.getcwd(), "data")

def load_json(filepath):
    full_path = os.path.join(DATA_DIR, filepath)
    if not os.path.exists(full_path):
        print(f"Warning: {full_path} not found.")
        return {}
    with open(full_path, "r", encoding="utf-8") as f:
        return json.load(f)

role_data = load_json("career_knowledge/mern_developer.json")
print(f"Loaded career data for role: {role_data.get('role', 'unknown')}")

---
## 4. Build LangGraph Interview Nodes

In [ ]:
INTERVIEWER_PERSONA = (
    "You are a professional interviewer at a mid-size tech company. "
    "Be professional but friendly. Ask follow-up questions when answers are vague. "
    "Your goal is to assess the candidate\'s fit for the role."
)

TECHNICAL_INSTRUCTIONS = (
    "Ask role-specific technical questions. Evaluate depth of knowledge, "
    "problem-solving approach, and familiarity with the tech stack. "
    "Ask follow-up questions that probe deeper into their understanding. "
    "Do not ask more than 2-3 questions per topic area."
)

HR_INSTRUCTIONS = (
    "Ask behavioral and situational questions. Evaluate communication skills, "
    "teamwork, conflict resolution, and cultural fit. "
    "Use the STAR format (Situation, Task, Action, Result) for questions."
)

BEHAVIORAL_INSTRUCTIONS = (
    "Present realistic workplace scenarios and ask how the candidate would handle them. "
    "Evaluate problem-solving, decision-making, and interpersonal skills. "
    "Probe for specific examples from their past experience."
)

CONCLUSION_INSTRUCTIONS = (
    "After 8-12 exchanges, conclude the interview naturally. "
    "Thank the candidate and tell them they will receive feedback shortly. "
    "Set next_action to \"conclude\" when ready to end."
)

print("System prompts defined")

In [ ]:
def process_turn_node(state: InterviewState) -> dict:
    """Process the current turn: handle student message, generate AI response."""
    interview_type = state["interview_type"]
    role = state["role"]
    history = list(state["conversation_history"])
    user_msg = state.get("user_message", "").strip()
    turn_count = state["turn_count"]
    sim_context = state.get("simulation_context", "")

    # Append student message to history if non-empty
    if user_msg:
        history.append({"role": "student", "content": user_msg})

    # Build system prompt
    type_instructions = TECHNICAL_INSTRUCTIONS
    if interview_type == "hr":
        type_instructions = HR_INSTRUCTIONS
    elif interview_type == "behavioral":
        type_instructions = BEHAVIORAL_INSTRUCTIONS

    system_lines = [
        INTERVIEWER_PERSONA,
        "",
        "## Role Being Interviewed For",
        role,
        "",
        "## Interview Type",
        interview_type.upper(),
        "",
        "## Interviewing Guidelines",
        type_instructions,
        "",
    ]

    if sim_context:
        system_lines.append("## Simulation Context (Candidate Background)")
        system_lines.append(sim_context)
        system_lines.append("")

    system_lines.append("## Conversation Flow Rules")
    system_lines.append("- If this is the start (no previous messages), open with a greeting and first question.")
    system_lines.append("- If the candidate has answered, acknowledge their response and ask a follow-up or new question.")
    system_lines.append("- Be specific. Reference what the candidate just said.")
    system_lines.append("- If the candidate gives a very short/vague answer, probe for more detail.")
    system_lines.append("- If the candidate gives a long answer, summarize and move to the next topic.")
    system_lines.append("")
    system_lines.append("## Turn Management")
    system_lines.append(f"Current turn: {turn_count + 1}. You should aim for 8-12 total exchanges before concluding.")
    system_lines.append("Set next_action to \"conclude\" when the interview has run its course.")
    system_lines.append("")
    system_lines.append("## Output Format")
    system_lines.append("Respond with your interview question or comment as ai_response.")
    system_lines.append("Set is_complete to True and next_action to \"conclude\" when ending the interview.")

    system_prompt = "\n".join(system_lines)

    # Build message list from history
    messages = [SystemMessage(content=system_prompt)]
    for entry in history:
        if entry["role"] == "student":
            messages.append(HumanMessage(content=entry["content"]))
        elif entry["role"] == "ai":
            messages.append(AIMessage(content=entry["content"]))

    # If no history yet, just send system prompt to get opening question
    try:
        response = model.invoke(messages)
        ai_text = response.content.strip()

        # Append AI response to history
        history.append({"role": "ai", "content": ai_text})

        return {
            "ai_response": ai_text,
            "conversation_history": history,
            "turn_count": turn_count + 1,
            "error": None,
        }
    except Exception as e:
        print(f"  [process_turn] Failed: {e}")
        return {"error": str(e)}

print("process_turn_node defined")

In [ ]:
def check_completion_node(state: InterviewState) -> dict:
    """Check if the interview should conclude based on AI response or turn count."""
    ai_text = state.get("ai_response", "")
    turn_count = state["turn_count"]
    history = state["conversation_history"]

    # Extract the number of student responses (exchanges)
    student_turns = sum(1 for e in history if e.get("role") == "student")

    # Heuristic: AI mentions concluding, or we have enough exchanges
    concluding_phrases = [
        "thank you, that concludes",
        "this concludes our interview",
        "thank you for your time",
        "will receive your feedback",
        "that concludes the interview",
        "i will be in touch",
        "thank you again for",
        "concludes our interview",
        "all the questions i had",
        "that is all the questions",
        "next_action",
        "is_complete = true",
    ]

    ai_lower = ai_text.lower()
    ai_concludes = any(phrase in ai_lower for phrase in concluding_phrases)

    # Conclude if AI signaled it, or we have enough turns (8+ student answers)
    should_conclude = ai_concludes or student_turns >= 10

    if should_conclude:
        print(f"  [check_completion] Interview complete after {student_turns} student exchanges")
        return {
            "is_complete": True,
            "next_action": "conclude",
        }
    else:
        return {
            "is_complete": False,
            "next_action": "continue",
        }

print("check_completion_node defined")

In [ ]:
def generate_report_node(state: InterviewState) -> dict:
    """Generate final scores, feedback, and interview summary from the full transcript."""
    history = state["conversation_history"]
    turn_count = state["turn_count"]
    interview_type = state["interview_type"]
    role = state["role"]

    # Format transcript for scoring
    transcript_lines = []
    for entry in history:
        role_label = "Candidate" if entry["role"] == "student" else "Interviewer"
        transcript_lines.append(f"{role_label}: {entry['content']}")
        transcript_lines.append("")
    transcript = "\n".join(transcript_lines)

    # Build scoring prompt
    score_categories = {
        "technical": "technical_knowledge",
        "communication": "communication",
        "confidence": "confidence",
        "problem_solving": "problem_solving",
        "cultural_fit": "cultural_fit",
    }

    prompt_lines = [
        "You are an expert interview evaluator. Review the following interview transcript"
        "and score the candidate in these categories (0-100 each):",
        "",
        "Scoring Categories:",
        "  technical_knowledge: Depth of technical knowledge and expertise",
        "  communication: Clarity, articulation, and professionalism of responses",
        "  confidence: Confidence level and decisiveness in answers",
        "  problem_solving: Analytical thinking and solution approach",
        "  cultural_fit: Alignment with company values and team dynamics",
        "",
        f"Interview Type: {interview_type}",
        f"Role: {role}",
        f"Total exchanges: {turn_count}",
        "",
        "## Transcript",
        transcript,
        "",
        "## Output Requirements",
        "Respond with a valid JSON object containing:",
        "  - scores: { technical_knowledge: int, communication: int, confidence: int, problem_solving: int, cultural_fit: int }",
        "  - overall_score: int (weighted average of all scores)",
        "  - feedback: string (detailed evaluation feedback 3-5 paragraphs)",
        "  - interview_summary: string (2-3 sentence summary of overall performance)",
        "",
        "Ensure all scores are integers between 0 and 100.",
        "Provide honest, constructive feedback that helps the candidate improve.",
    ]

    prompt = "\n".join(prompt_lines)

    try:
        response = model.bind(response_format={"type": "json_object"}).invoke([
            SystemMessage(content=prompt),
        ])
        content = response.content

        report = json.loads(content)

        required = ["scores", "overall_score", "feedback", "interview_summary"]
        for key in required:
            if key not in report:
                raise ValueError(f"Report missing required field: {key}")

        score_keys = ["technical_knowledge", "communication", "confidence",
                      "problem_solving", "cultural_fit"]
        for key in score_keys:
            if key not in report.get("scores", {}):
                raise ValueError(f"Report missing score field: {key}")

        print(f"  [generate_report] Overall score: {report['overall_score']}")
        return {
            "scores": report["scores"],
            "overall_score": report["overall_score"],
            "feedback": report["feedback"],
            "interview_summary": report["interview_summary"],
            "is_complete": True,
            "next_action": "conclude",
        }
    except Exception as e:
        print(f"  [generate_report] Failed: {e}")
        return {"error": str(e)}

print("generate_report_node defined")

---
## 5. Compile the LangGraph

In [ ]:
def build_interview_graph():
    workflow = StateGraph(InterviewState)

    workflow.add_node("process_turn", process_turn_node)
    workflow.add_node("check_completion", check_completion_node)
    workflow.add_node("generate_report", generate_report_node)

    workflow.set_entry_point("process_turn")

    workflow.add_edge("process_turn", "check_completion")

    def route_from_completion(state):
        if state.get("next_action") == "conclude" or state.get("is_complete"):
            return "generate_report"
        return END

    workflow.add_conditional_edges("check_completion", route_from_completion, {
        "generate_report": "generate_report",
        END: END,
    })

    workflow.add_edge("generate_report", END)

    return workflow.compile()

app = build_interview_graph()
print("LangGraph Interview Agent compiled successfully")

---
## 6. Helper Functions

In [ ]:
def run_interview_turn(
    interview_type: str,
    role: str,
    conversation_history: list,
    user_message: str = "",
    simulation_context: str = "",
    max_turns: int = 15,
) -> dict:
    """Run a single turn of the interview. Returns updated state."""
    state = InterviewState(
        interview_type=interview_type,
        role=role,
        conversation_history=conversation_history,
        simulation_context=simulation_context,
        user_message=user_message,
        ai_response="",
        turn_count=0,
        max_turns=max_turns,
        is_complete=False,
        next_action="continue",
        scores=None,
        overall_score=None,
        feedback=None,
        interview_summary=None,
        error=None,
    )

    try:
        result = app.invoke(state)
        return result
    except Exception as e:
        print(f"Interview turn failed: {e}")
        return {"error": str(e)}

print("run_interview_turn function defined")

In [ ]:
def display_interaction(state: dict):
    if state.get("error"):
        error_msg = state["error"]
        print(f"[ERROR] {error_msg}")
        return

    print("=" * 70)
    print("INTERVIEW SESSION")
    print("=" * 70)
    print(f"Type: {state.get('interview_type', 'N/A').upper()}")
    print(f"Role: {state.get('role', 'N/A')}")
    print(f"Turns: {state.get('turn_count', 0)}")
    print(f"Complete: {state.get('is_complete', False)}")
    print(f"Action: {state.get('next_action', 'N/A')}")

    # Show conversation
    history = state.get("conversation_history", [])
    for entry in history:
        label = "\n[AI INTERVIEWER]" if entry["role"] == "ai" else "\n[CANDIDATE]"
        print(f"{label}")
        print(f"{entry.get('content', '')}")

    # Show scores if available
    scores = state.get("scores")
    if scores:
        print(f"\n{'=' * 70}")
        print("FINAL SCORES")
        print(f"{'=' * 70}")

        categories = [
            ("Technical Knowledge", "technical_knowledge"),
            ("Communication", "communication"),
            ("Confidence", "confidence"),
            ("Problem Solving", "problem_solving"),
            ("Cultural Fit", "cultural_fit"),
        ]
        for label, key in categories:
            score = scores.get(key, "N/A")
            bar = "#" * (score // 5) if isinstance(score, int) else ""
            print(f"  {label:20s}: {score:>3d} {bar}")

        print(f"\nOVERALL SCORE: {state.get('overall_score', 'N/A')}/100")

        print(f"FEEDBACK:\n{state.get('feedback', 'N/A')}")
        print(f"\nSUMMARY:\n{state.get('interview_summary', 'N/A')}")

print("display_interaction function defined")

In [ ]:
def simulate_interview(
    interview_type: str,
    role: str,
    student_answers: list,
    simulation_context: str = "",
    verbose: bool = True,
) -> dict:
    """Simulate a full interview with pre-written student answers."""
    history = []
    final_state = None

    # Step 1: Start the interview (empty user_message triggers first question)
    if verbose:
        print("\n" + "=" * 70)
        print(f"STARTING {interview_type.upper()} INTERVIEW")
        print(f"Role: {role}")
        print("=" * 70)

    state = run_interview_turn(
        interview_type=interview_type,
        role=role,
        conversation_history=history,
        user_message="",
        simulation_context=simulation_context,
    )

    if state.get("error"):
        print(f"Failed to start interview: {state["error"]}")
        return state

    if verbose:
        history = state.get("conversation_history", [])
        if history:
            print(f"\n[AI INTERVIEWER]\n{history[-1].get('content', '')}")

    # Step 2: Process each student answer
    for i, answer in enumerate(student_answers):
        state = run_interview_turn(
            interview_type=interview_type,
            role=role,
            conversation_history=state.get("conversation_history", []),
            user_message=answer,
            simulation_context=simulation_context,
        )

        if state.get("error"):
            print(f"Turn {i+1} failed: {state["error"]}")
            return state

        if verbose:
            print(f"\n[CANDIDATE]\n{answer}")
            history = state.get("conversation_history", [])
            if history:
                print(f"\n[AI INTERVIEWER]\n{history[-1].get('content', '')}")

        # Check if interview concluded
        if state.get("next_action") == "conclude" or state.get("is_complete"):
            break

    # Step 3: Display final report if available
    scores = state.get("scores")
    if verbose and scores:
        print(f"\n{'=' * 70}")
        print("INTERVIEW COMPLETE - FINAL REPORT")
        print(f"{'=' * 70}")
        print(f"\nOverall Score: {state.get('overall_score', 'N/A')}/100")

        for cat, val in scores.items():
            print(f"  {cat.replace('_', ' ').title():25s}: {val}")

        print(f"\nFEEDBACK:\n{state.get('feedback', 'N/A')}")
        print(f"\nSUMMARY:\n{state.get('interview_summary', 'N/A')}")

    return state

print("simulate_interview function defined")

---
## 7. Tests

### Test 1: Full 10-turn technical interview

In [ ]:
print("=" * 70)
print("TEST 1: Full Technical Interview (MERN Stack Developer)")
print("=" * 70)

student_answers_tech = [
    "I have about 2 years of experience with the MERN stack. I have built several full-stack apps including an e-commerce platform and a task management tool.",
    "I think the most challenging part was implementing real-time notifications with Socket.io while keeping the MongoDB queries optimized. We had to use aggregation pipelines to reduce the load.",
    "For authentication, I usually implement JWT with refresh tokens. The access token expires in 15 minutes and the refresh token in 7 days. I store the refresh token in an httpOnly cookie.",
    "I use indexes on frequently queried fields, and I have used MongoDB aggregation framework for complex reporting queries. I also make sure to use projection to limit returned fields.",
    "I follow the MVC pattern - models with Mongoose schemas, controllers for business logic, and routes for endpoint definitions. I use middleware for auth validation and error handling.",
    "I write unit tests with Jest for utility functions and integration tests with Supertest for API endpoints. I aim for at least 80% coverage on critical paths.",
    "For state management, I use React Query for server state and React Context for UI state. I find this combination keeps the code clean and avoids prop drilling.",
    "In production, I use environment-specific config files, centralized error handling with a custom error middleware, and I log errors to a monitoring service.",
    "I think clean code is about readability and maintainability. I follow consistent naming conventions, keep functions small, and document complex logic.",
    "I resolve conflicts by first understanding the other person perspective, then suggesting a compromise or a quick A/B test to settle the decision objectively.",
]

sim_context = (
    "Completed 5 simulation days: fixed JWT auth bug, built data table component, "
    "implemented file upload, designed user profile API, and wrote integration tests. "
    "Demonstrated strong problem-solving and code quality skills."
)

result = simulate_interview(
    interview_type="technical",
    role="MERN Stack Developer",
    student_answers=student_answers_tech,
    simulation_context=sim_context,
)

if result.get("scores"):
    print("\nPASS: Technical interview completed with scores")
else:
    print("\nWARNING: Interview did not produce scores (may have been interrupted)")

### Test 2: Full HR interview

In [ ]:
print("=" * 70)
print("TEST 2: HR Interview - Behavioral & Cultural Fit")
print("=" * 70)

student_answers_hr = [
    "I chose software development because I enjoy building things that help people. In my last role, I built an internal tool that saved the team 10 hours per week.",
    "My biggest strength is adaptability - I can switch between frontend and backend work easily. My weakness is that I sometimes spend too much time perfecting code instead of shipping.",
    "In a previous team, we had a disagreement about using Redux vs Context API. I suggested we prototype both and compare, which helped us make an objective decision.",
    "I thrive in teams with clear communication and regular code reviews. I value when people give constructive feedback respectfully.",
    "Under pressure, I prioritize by impact. I focus on what's most critical for the user and communicate early if timelines need adjustment.",
    "I once made a mistake that caused a production outage for 30 minutes. I took ownership, fixed it immediately, and wrote a post-mortem to prevent it from happening again.",
    "I stay updated by following tech blogs, contributing to open-source projects, and building side projects to experiment with new technologies.",
    "I see myself growing into a senior developer role where I can mentor juniors and make architectural decisions for the team.",
]

result = simulate_interview(
    interview_type="hr",
    role="MERN Stack Developer",
    student_answers=student_answers_hr,
    simulation_context="",
)

if result.get("scores"):
    print("\nPASS: HR interview completed with scores")
else:
    print("\nWARNING: HR interview did not produce scores (may have been interrupted)")

### Test 3: One-word answers — check AI probes deeper

In [ ]:
print("=" * 70)
print("TEST 3: One-Word Answers - AI should probe deeper")
print("=" * 70)

one_word_answers = [
    "Yes.",
    "React.",
    "Fine.",
    "Sometimes.",
    "Maybe.",
    "Hard.",
    "Good.",
    "Okay.",
]

result = simulate_interview(
    interview_type="technical",
    role="MERN Stack Developer",
    student_answers=one_word_answers,
    verbose=False,
)

history = result.get("conversation_history", [])
follow_ups = 0
for i, entry in enumerate(history):
    if entry["role"] == "ai" and i > 0:
        prev = history[i-1] if i > 0 else None
        if prev and prev["role"] == "student" and len(prev["content"].strip().split()) <= 3:
            follow_ups += 1

print(f"\nAI follow-ups after short answers: {follow_ups}")
if follow_ups >= 3:
    print("PASS: AI asked follow-up questions when answers were too short")
else:
    print("WARNING: AI could have probed deeper on short answers")

### Test 4: Very long answers — check AI summarizes and moves on

In [ ]:
print("=" * 70)
print("TEST 4: Very Long Answers - AI should summarize and move on")
print("=" * 70)

long_answers = [
    "Let me tell you about a project I worked on. It was an e-commerce platform built with React, Node.js, and MongoDB. The project took about 6 months. We had a team of 5 people. I was responsible for the frontend architecture and part of the API design. We used Redux for state management initially but later migrated to React Query. The authentication was implemented using JWT tokens with refresh token rotation. We also implemented payment integration with Stripe. The project had about 50,000 monthly active users. We used AWS for hosting with EC2 and S3 for file storage. I learned a lot about performance optimization and caching strategies during this project. One of the key challenges was handling real-time inventory updates across multiple server instances, which we solved using Redis pub/sub. Overall it was a great learning experience.",
    "Regarding testing, I believe in a comprehensive testing strategy. I write unit tests for all utility functions and helpers, integration tests for API endpoints, and end-to-end tests for critical user flows. I use Jest as my test runner along with React Testing Library for component tests. For API testing, I prefer Supertest as it allows me to test endpoints without actually starting the server. I also use Cypress for E2E testing of the most critical paths like user registration and checkout flow. In my last project, we achieved 85% code coverage. I also believe in testing error scenarios and edge cases, not just happy paths. I find that good tests give me confidence when refactoring code and help catch regressions early. Testing is not just about coverage numbers though - I focus on testing behavior rather than implementation details to make tests more maintainable.",
    "For CI/CD, we used GitHub Actions with a pipeline that ran linting, type checking, unit tests, and integration tests on every pull request. The pipeline also built the Docker image and deployed to staging for review apps. Once merged to main, it would automatically deploy to production after manual approval. We used feature flags to gradually roll out new features to a percentage of users. This helped us catch issues early before they affected all users. We also had automated performance testing using Lighthouse CI to ensure our changes did not degrade performance. The deployment process was fully automated except for the production deployment step which required a senior developers approval. We monitored the application using Sentry for error tracking and Datadog for infrastructure monitoring.",
]

result = simulate_interview(
    interview_type="technical",
    role="MERN Stack Developer",
    student_answers=long_answers,
    verbose=False,
)

history = result.get("conversation_history", [])
ai_responses = [e["content"] for e in history if e["role"] == "ai"]

print(f"\nAI responses: {len(ai_responses)}")
avg_ai_len = sum(len(r.split()) for r in ai_responses) / max(len(ai_responses), 1)
print(f"Average AI response length: {avg_ai_len:.0f} words")
if avg_ai_len < 150:
    print("PASS: AI kept responses concise when candidate gave long answers")
else:
    print("WARNING: AI responses were long despite candidate giving detailed answers")

### Test 5: Verify final score generation with mock transcript

In [ ]:
print("=" * 70)
print("TEST 5: Score Generation Verification")
print("=" * 70)

mock_history = [
    {"role": "student", "content": "I have 3 years of React experience."},
    {"role": "ai", "content": "Tell me about your experience with state management."},
    {"role": "student", "content": "I use Redux Toolkit and React Query extensively. I prefer React Query for server state and Redux for complex client state."},
    {"role": "ai", "content": "How do you handle side effects?"},
    {"role": "student", "content": "I use useEffect for simple side effects and custom hooks for complex ones. I also use React Query mutations for server mutations."},
    {"role": "ai", "content": "What testing strategy do you follow?"},
    {"role": "student", "content": "I write unit tests with Jest, integration tests with React Testing Library, and E2E with Cypress."},
    {"role": "ai", "content": "How do you optimize React performance?"},
    {"role": "student", "content": "I use React.memo for expensive components, useCallback for callback props, and virtualization for long lists."},
]

state = InterviewState(
    interview_type="technical",
    role="MERN Stack Developer",
    conversation_history=mock_history,
    simulation_context="",
    user_message="",
    ai_response="",
    turn_count=4,
    max_turns=15,
    is_complete=True,
    next_action="conclude",
    scores=None,
    overall_score=None,
    feedback=None,
    interview_summary=None,
    error=None,
)

print("Generating report from mock transcript...")
report_state = generate_report_node(state)

if report_state.get("error"):
    print(f"FAIL: {report_state['error']}")
else:
    scores = report_state.get("scores", {})
    overall = report_state.get("overall_score", 0)
    feedback = report_state.get("feedback", "")
    summary = report_state.get("interview_summary", "")

    print(f"\nScores: {scores}")
    print(f"Overall: {overall}")
    print(f"Feedback present: {bool(feedback)}")
    print(f"Summary present: {bool(summary)}")

    all_present = all(k in scores for k in
        ["technical_knowledge", "communication", "confidence",
         "problem_solving", "cultural_fit"])
    if all_present and overall > 0 and feedback and summary:
        print("PASS: All required fields present in report")
    else:
        print("FAIL: Missing required fields in report")

---
## 8. Summary

### Test Results Checklist
- [ ] **Test 1:** Full 10-turn technical interview — check question quality and follow-ups
- [ ] **Test 2:** Full HR interview — check for behavioral question coverage
- [ ] **Test 3:** One-word answers — check that AI probes deeper
- [ ] **Test 4:** Very long answers — check AI summarizes and moves on
- [ ] **Test 5:** Verify final score is generated correctly from transcript

### Architecture
```
StateGraph(InterviewState)
  [process_turn] -> [check_completion]
                          |
            ┌─────────────┼─────────────┐
            v             v             v
     next_action=    next_action=    error
      'continue'      'conclude'
          |                |
        END      [generate_report] -> END
```

### Scoring Categories
| Category | Description |
|----------|-------------|
| technical_knowledge | Depth of technical expertise |
| communication | Clarity and articulation |
| confidence | Confidence in responses |
| problem_solving | Analytical thinking |
| cultural_fit | Team and culture alignment |

### AI Model
- `llama-3.1-8b-instant` via Groq (fast, cost-efficient for multi-turn chat)
- `ChatGroq` with JSON mode for structured report generation